# Mixed Local-Evidence Cross-Dataset Evaluation

Train local-evidence classifiers on mixed Kaggle, Outfox, and MAGE data, then evaluate transfer on each dataset separately for Gemma 3 1B and SmolLM 360M.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from local_evidence_evaluation_utils import (
    evaluate_local_evidence_datasets,
    evaluate_local_evidence_semeval,
)
from local_evidence_training_utils import train_local_evidence_models
from local_evidence_models_utils import (
    BayesianConfig,
    GaussianMixtureNBConfig,
    LOCAL_FEATURE_COLUMNS,
)
from local_evidence_utils import (
    load_or_build_local_evidence_windows,
    split_local_evidence_texts,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

In [ ]:
DATASETS = {
    "kaggle_train": "Kaggle",
    "outfox_chatgpt_train": "Outfox",
    "mage_train": "MAGE",
}
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_dev"
SEMEVAL_SPLIT = "MG"
SEMEVAL_PREPARED_PATH = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_dev_MG.csv"
SEMEVAL_MG_SCORE_THRESHOLD = 0.5
SEMEVAL_SMOOTHING_WINDOW = 5
SEMEVAL_MINIMUM_REGION_LENGTH = 3
MODELS = ["gemma_3_1b_it", "smollm2_360m"]
WINDOW_LENGTHS = (32, 64, 128)
STRIDE = 16
TOP_K = 20
SEED = 42
RECOMPUTE_WINDOWS = False
REUSE_FITTED_MODELS = True
SAVE_FITTED_MODELS = False
TRUST_FITTED_MODEL_CACHE = False
MIXED_REGIME = "Mixed"
BAYESIAN_CONFIG = BayesianConfig(advi_iterations=10_000, posterior_draws=1_000, random_seed=SEED)
GAUSSIAN_MIXTURE_CONFIG = GaussianMixtureNBConfig(components=2, reg_covar=1e-6)
try:
    import pymc
except ModuleNotFoundError:
    PYMC_AVAILABLE = False
else:
    PYMC_AVAILABLE = True
if not PYMC_AVAILABLE:
    display("PyMC unavailable; continuing without pymc_bayesian_logistic. Install it with: uv sync --extra bayesian")
REQUESTED_MODELS = ("logistic_regression", "gaussian_nb", "gaussian_mixture_nb", "pymc_bayesian_logistic")
AVAILABLE_MODELS = tuple(name for name in REQUESTED_MODELS if name != "pymc_bayesian_logistic" or PYMC_AVAILABLE)

In [ ]:
result_frames = []
semeval_summary_frames = []
semeval_evaluation_frames = []
coverage_frames = []
for processing_model in MODELS:
    train_ids, validation_ids, test_ids = {}, {}, {}
    valid_datasets = []
    for dataset in DATASETS:
        try:
            windows = load_or_build_local_evidence_windows(
                PROCESSED_ROOT, dataset, processing_model, window_lengths=WINDOW_LENGTHS,
                stride=STRIDE, top_k=TOP_K, recompute=RECOMPUTE_WINDOWS,
            )
            partitions = split_local_evidence_texts(windows, seed=SEED)
        except (FileNotFoundError, ValueError) as error:
            display(f"Skipping {DATASETS[dataset]}/{processing_model}: {error}")
            continue
        train_ids[dataset] = partitions["train"]
        validation_ids[dataset] = partitions["validation"]
        test_ids[dataset] = partitions["test"]
        valid_datasets.append(dataset)
        coverage_frames.append(windows.groupby("split").size().rename("rows").reset_index().assign(processing_model=processing_model, source_dataset=DATASETS[dataset]))
    if not valid_datasets:
        continue
    trained = train_local_evidence_models(
        PROCESSED_ROOT, valid_datasets, processing_model, train_ids, validation_ids,
        classifier_names=AVAILABLE_MODELS, window_lengths=WINDOW_LENGTHS,
        feature_columns=LOCAL_FEATURE_COLUMNS, stride=STRIDE, top_k=TOP_K, seed=SEED,
        bayesian_config=BAYESIAN_CONFIG, gaussian_mixture_config=GAUSSIAN_MIXTURE_CONFIG,
    )
    for evaluation_dataset in valid_datasets:
        evaluated = evaluate_local_evidence_datasets(
            trained.model, PROCESSED_ROOT, [evaluation_dataset],
            {evaluation_dataset: test_ids[evaluation_dataset]},
        )
        result_frames.append(evaluated.metrics.assign(processing_model=processing_model, classifier=evaluated.metrics["model"]))
    semeval_annotations = pd.read_csv(SEMEVAL_PREPARED_PATH)
    if {"id", "text", "boundary_word_index"}.difference(semeval_annotations.columns):
        raise ValueError("Prepared SemEval CSV must contain id, text, and boundary_word_index columns.")
    semeval = evaluate_local_evidence_semeval(
        trained.model, PROCESSED_ROOT, semeval_annotations, dataset=SEMEVAL_DATASET, split=SEMEVAL_SPLIT,
        threshold=SEMEVAL_MG_SCORE_THRESHOLD, smoothing_window=SEMEVAL_SMOOTHING_WINDOW,
        minimum_region_length=SEMEVAL_MINIMUM_REGION_LENGTH,
    )
    if not semeval.summary.empty:
        semeval_summary_frames.append(semeval.summary.assign(processing_model=processing_model))
        semeval_evaluation_frames.append(semeval.evaluations.assign(processing_model=processing_model))
        display(semeval.summary.sort_values(["mg_f1", "model", "window_length"]))
coverage = pd.concat(coverage_frames, ignore_index=True) if coverage_frames else pd.DataFrame()
display(coverage.sort_values(["processing_model", "source_dataset", "split"]))
if not result_frames:
    raise ValueError("No mixed local-evidence training or evaluation results are available.")
results = pd.concat(result_frames, ignore_index=True)
display(results.sort_values(["processing_model", "model", "window_length", "evaluation_dataset"]))
if semeval_summary_frames:
    semeval_region_summary = pd.concat(semeval_summary_frames, ignore_index=True)
    semeval_region_evaluations = pd.concat(semeval_evaluation_frames, ignore_index=True)
    display(semeval_region_summary.sort_values(["processing_model", "mg_f1", "model", "window_length"]))
else:
    display("No SemEval region evaluations are available.")

In [ ]:
for processing_model in MODELS:
    model_results = results.query(
        "processing_model == @processing_model and aggregation == 'evidence_fraction'"
    ).copy()
    if model_results.empty:
        continue
    matrix = (
        model_results.pivot(index=["classifier", "window_length"], columns="evaluation_dataset", values="macro_f1")
        .reindex(columns=["Kaggle", "Outfox", "MAGE"])
    )
    display(matrix)
    plt.figure(figsize=(7, max(3, len(matrix) * 0.45)), constrained_layout=True)
    sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0.0, vmax=1.0, cmap="viridis", mask=matrix.isna())
    plt.title(f"Mixed-training macro-F1 by evaluation dataset: {processing_model}")
    plt.xlabel("Evaluation dataset")
    plt.ylabel("Classifier / window length")
    plt.show()